In [ ]:
import numpy as np
from tqdm import tqdm
from itertools import combinations

In [ ]:
# Input parameters
P = int(input("Enter number of parties (P): "))  # Number of parties
I = int(input("Enter number of interventions (I): "))  # Number of interventions per party
O = int(input("Enter number of outcomes (O): "))  # Number of outcomes per party

In [ ]:
# Helper function to generate output combinations (O^P)
def OPlist():
    OP = []
    for i in range(O**P):
        changedBase = 0
        temp = i
        count = 0
        while temp > 0:
            rem = temp % O
            changedBase += rem * (10**count)
            count += 1
            temp = temp // O
        OP += [(P - len(str(changedBase))) * '0' + str(changedBase)]
    return OP

# Helper function to generate input combinations (I^P)
def IPlist():
    IP = []
    for i in range(I**P):
        changedBase = 0
        temp = i
        count = 0
        while temp > 0:
            rem = temp % I
            changedBase += rem * (10**count)
            count += 1
            temp = temp // I
        IP += [(P - len(str(changedBase))) * '0' + str(changedBase)]
    return IP

In [ ]:
# Function to change number base from base frombase to another base. m is padding
def ChangeBase2(num, frombase, tobase, m):
    changedBase = 0
    temp = num
    count = 0
    while temp > 0:
        rem = temp % tobase
        changedBase += rem * (frombase**count)
        count += 1
        temp = temp // tobase
    return (m - len(str(changedBase))) * '0' + str(changedBase)

# Function to change number base from base 10 to another base
def ChangeBase(num, tobase, m):
    changedBase = 0
    temp = num
    count = 0
    while temp > 0:
        rem = temp % tobase
        changedBase += rem * (10**count)
        count += 1
        temp = temp // tobase
    return (m - len(str(changedBase))) * '0' + str(changedBase)

# Function to change number base (as list)
def ChangeBaseList(num, tobase, m):
    changedBase = []
    temp = num
    count = 0
    while temp > 0:
        rem = temp % tobase
        changedBase += [rem]
        count += 1
        temp = temp // tobase
    temp=[]
    for i in range(m - len(changedBase)):
        temp+=[0]
    changedBase += temp
    return changedBase

# Initialize empty table
def Tables(OP, IP):
    TD = [[0 for _ in range(len(IP))] for _ in range(len(OP))]
    return TD

#Function to print table
def PrintTable(TD, OP, IP):
    print("OP\\IP", end='\t')
    for ip in IP:
        print(ip, end='\t')
    print()
    for i, op in enumerate(OP):
        print(op, end='\t')
        for j in range(len(IP)):
            print(TD[i][j], end='\t')
        print()

In [ ]:
# Generic PNS checker
def check_possibilistic_no_signaling(Table, P, I, O):
    for i in range(P): #fixes (a,x) or (b,y) etc
        #print("Party:",i)
        for j in range(I): #fixes x or y etc
            for k in range(O): # fixes a or b etc
                MarginalList=[]
                for l in range(I): #fixes y or x resp. 
                    if i == 0: # if first party, then a
                        column= int(ChangeBase2(int(str(j)+str(l)),I,10,2))
                    else: # if not first party, then b
                        column= int(ChangeBase2(int(str(l)+str(j)),I,10,2))

                    Rows=[]
                    for temp in range(O): # takes all b or a resp.
                        if i == 0:
                            index= int(ChangeBase2(int(str(k)+str(temp)),O,10,2))
                            Rows+=[index]
                        else: # if not first party, then b
                            index= int(ChangeBase2(int(str(temp)+str(k)),O,10,2))
                            Rows+=[index]
                   #print("Rows:", Rows, "Column:", column)
                    marginal=0
                    for row in Rows:
                        marginal = (marginal or Table[row][column])
                        #marginal = (marginal or Table[column][row])
                    MarginalList+=[marginal]

                #print("MarginalList:", MarginalList)
                if (0 in MarginalList) and (1 in MarginalList):
                    return False          
    return True
                    


In [ ]:
# Generate OP and IP lists
OP = OPlist()
IP = IPlist()

# Base FD mappings
fd = [[1 if i == j else 0 for j in range(O**P)] for i in range(O**P)]
#print("All FD configurations for a single combination of intervention:")
#print(fd)

# Generate all possible PNS configurations for a single combination of intervention
pns = []
num_outputs = O**P  # Total number of output configurations
for i in range(1,2**num_outputs):  # 2 options for each output (possible/not possible)
    changed_base = ChangeBase(i, tobase=2, m=num_outputs)
    pns.append([int(digit) for digit in changed_base])
#print("All PNS configurations for a single combination of intervention:")
#print(pns)

# Generate all FD codes
AllFDCode = [i for i in range(len(fd))]
#print(AllFDCode) AllFDCode is a list of numbers from 0 to O**P

# Generate all possible PNS tables across all interventions
AllPNSCode = []
N = (2**num_outputs-1)**((I**P)-1)  # Total number of PNS tables
n = 2**num_outputs-1
m = (I**P)-1

# Generate all codes (this is time-intensive; vectorization used later)
print("Generating all PNS Codes...")
for i in tqdm(range(N)):
    changed_base = ChangeBaseList(i, n, m)
    AllPNSCode.append(changed_base)
#print(AllPNSCode) 
#AllPNSCode is a list of strings of size m, each is a number in base n from 0 to N

AllPDCode=[]
# Generate all possible PD tables codes
print("Generating all PD Codes...")
for i in range(len(fd)):
    for j in tqdm(range(N)):
        code=[AllFDCode[i]]
        code+=[k for k in AllPNSCode[j]]
        #print(code)
        AllPDCode.append(code)


In [ ]:
# Validate PNS tables for Possibilistic No-Signaling
AllPDTables = []
print("Validating PNS tables...")
for code in tqdm(AllPDCode):
    # Convert the code into a matrix
    PD_table_transpose = [fd[int(code[0])]]
    PNSpartCode=code[1:]
    PD_table_transpose += [pns[int(c)] for c in PNSpartCode]
    
    # Transpose the table to its correct structure
    PD_table = [
        [PD_table_transpose[j][i] for j in range(len(PD_table_transpose))]
        for i in range(len(PD_table_transpose[0]))
    ]
    
    #print(PD_table)
    
    # Check if the table satisfies Possibilistic No-Signaling
    if check_possibilistic_no_signaling(PD_table, P, I, O):
        AllPDTables.append(PD_table)

# Results
print(f"Number of PD tables satisfying PNS: {len(AllPDTables)}")
print(f"Total number of PD tables: {len(AllPDCode)}")
for table in AllPDTables:
    #print(table)
    PrintTable(table, OP, IP)

# ORs of PD tables

In [ ]:
import itertools

def powerset(iterable):
    """
    Generates all non-empty subsets (as tuples) of the given iterable.
    For example, powerset([a, b, c]) returns:
      (a,), (b,), (c,), (a, b), (a, c), (b, c), (a, b, c)
    """
    s = list(iterable)
    # Generate combinations for all possible lengths (from 1 to len(s))
    return itertools.chain.from_iterable(itertools.combinations(s, r) for r in range(1, len(s) + 1))

def or_tables(tables):
    """
    Given a list of 2-D tables (each table is a list of lists with 0s and 1s),
    computes the element-wise OR of these tables.
    Assumes that all tables have the same dimensions.
    """
    rows = len(tables[0])           # Number of rows in the first table
    cols = len(tables[0][0])        # Number of columns in the first table
    # Initialize the result table with zeros (same dimensions as the input tables)
    result = [[0] * cols for _ in range(rows)]
    # For every table in the list:
    for table in tables:
        # For every row index i:
        for i in range(rows):
            # For every column index j:
            for j in range(cols):
                # Compute the OR: if either the current result or table value is 1, result becomes 1.
                result[i][j] = result[i][j] or table[i][j]
    return result

tables=AllPDTables

AllUniqueORs=[]
# Loop through every non-empty subset of tables
for subset in powerset(tables):
    # Calculate the entry-wise OR for the current subset
    combined = or_tables(subset)
    if combined not in AllUniqueORs:
        AllUniqueORs.append(combined)

#To print all subsets and results uncomment the below:
'''
    print("Subset:")
    # Print each table in the subset
    for table in subset:
        for row in table:
            print(row)
        print("-----")
    
    print("OR result:")
    # Print the resulting table after performing entry-wise OR on the subset
    for row in combined:
        print(row)
    print("=======\n")
'''
# Results
print(f"Number of unique OR tables: {len(AllUniqueORs)}")
#for i, table in enumerate(AllUniqueORs):
    #print(f"Table {i + 1}:\n{table}\n")

In [ ]:
print(f"Number of unique OR tables: {len(AllUniqueORs)}")